# บทที่ 5: Regularization Techniques

ใน Notebook นี้ เราจะเรียนรู้เทคนิคต่างๆ เพื่อป้องกัน Overfitting ได้แก่ L1/L2 Regularization, Dropout, Early Stopping, Batch Normalization และ Data Augmentation

## 1. นำเข้าไลบรารี

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

# ติดตั้งฟอนต์ภาษาไทยสำหรับ Google Colab
import subprocess, glob
subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-tlwg-garuda'], 
               capture_output=True)

# ลงทะเบียนฟอนต์โดยตรง
from matplotlib.font_manager import fontManager
for font_file in glob.glob('/usr/share/fonts/truetype/tlwg/*.ttf'):
    fontManager.addfont(font_file)

# ตั้งค่า Seaborn theme และฟอนต์ภาษาไทย
sns.set_theme(style='whitegrid', font='Garuda')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
%config InlineBackend.figure_format = 'retina'

from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split

np.random.seed(42)

## 2. สร้างข้อมูลตัวอย่าง

In [ ]:
# สร้างข้อมูล make_moons
X, y = make_moons(n_samples=500, noise=0.25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

plt.figure(figsize=(8, 6))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='coolwarm', alpha=0.7)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('ข้อมูล Make Moons')
plt.show()

## 3. L1 และ L2 Regularization

- **L1 (Lasso)**: $\lambda \sum |w_i|$ - ทำให้ weights บางตัวเป็น 0
- **L2 (Ridge)**: $\lambda \sum w_i^2$ - ลดขนาด weights ทุกตัว

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

class MLP_Regularized:
    """MLP with L1/L2 Regularization"""
    
    def __init__(self, layer_sizes, learning_rate=0.5, l1_lambda=0.0, l2_lambda=0.0):
        self.layer_sizes = layer_sizes
        self.lr = learning_rate
        self.l1_lambda = l1_lambda
        self.l2_lambda = l2_lambda
        
        self.weights = []
        self.biases = []
        
        for i in range(len(layer_sizes) - 1):
            w = np.random.randn(layer_sizes[i+1], layer_sizes[i]) * 0.5
            b = np.zeros((layer_sizes[i+1], 1))
            self.weights.append(w)
            self.biases.append(b)
            
    def forward(self, x):
        self.activations = [x.reshape(-1, 1)]
        self.z_values = []
        
        current = self.activations[0]
        for i in range(len(self.layer_sizes) - 1):
            z = np.dot(self.weights[i], current) + self.biases[i]
            self.z_values.append(z)
            current = sigmoid(z)
            self.activations.append(current)
            
        return current
    
    def backward(self, y):
        y = y.reshape(-1, 1)
        delta = self.activations[-1] - y
        
        self.dW = []
        self.db = []
        
        for i in range(len(self.layer_sizes) - 2, -1, -1):
            # Add regularization to gradient
            l1_grad = self.l1_lambda * np.sign(self.weights[i])
            l2_grad = self.l2_lambda * self.weights[i]
            
            dW = np.dot(delta, self.activations[i].T) + l1_grad + l2_grad
            db = delta
            
            self.dW.insert(0, dW)
            self.db.insert(0, db)
            
            if i > 0:
                delta = np.dot(self.weights[i].T, delta) * sigmoid_derivative(self.z_values[i-1])
                
    def update_weights(self):
        for i in range(len(self.layer_sizes) - 1):
            self.weights[i] -= self.lr * self.dW[i]
            self.biases[i] -= self.lr * self.db[i]
            
    def train(self, X, y, epochs=1000):
        for epoch in range(epochs):
            for xi, yi in zip(X, y):
                self.forward(xi)
                self.backward(yi)
                self.update_weights()
                
    def predict(self, x):
        return self.forward(x)[0, 0]
    
    def compute_loss(self, X, y):
        total = 0
        for xi, yi in zip(X, y):
            pred = self.predict(xi)
            total += (yi - pred)**2
        return total / len(X)

## 4. เปรียบเทียบ No Regularization vs L2 Regularization

In [ ]:
# Train without regularization
mlp_no_reg = MLP_Regularized([2, 64, 64, 1], learning_rate=0.5, l1_lambda=0, l2_lambda=0)
mlp_no_reg.train(X_train, y_train, epochs=2000)

# Train with L2 regularization
mlp_l2 = MLP_Regularized([2, 64, 64, 1], learning_rate=0.5, l1_lambda=0, l2_lambda=0.01)
mlp_l2.train(X_train, y_train, epochs=2000)

print("=== Training Loss ===")
print(f"No Regularization: {mlp_no_reg.compute_loss(X_train, y_train):.4f}")
print(f"L2 Regularization: {mlp_l2.compute_loss(X_train, y_train):.4f}")

print("\n=== Test Loss ===")
print(f"No Regularization: {mlp_no_reg.compute_loss(X_test, y_test):.4f}")
print(f"L2 Regularization: {mlp_l2.compute_loss(X_test, y_test):.4f}")

## 5. Dropout Implementation

In [ ]:
class MLP_Dropout:
    """MLP with Dropout"""
    
    def __init__(self, layer_sizes, learning_rate=0.5, dropout_rate=0.5):
        self.layer_sizes = layer_sizes
        self.lr = learning_rate
        self.dropout_rate = dropout_rate
        
        self.weights = []
        self.biases = []
        
        for i in range(len(layer_sizes) - 1):
            w = np.random.randn(layer_sizes[i+1], layer_sizes[i]) * 0.5
            b = np.zeros((layer_sizes[i+1], 1))
            self.weights.append(w)
            self.biases.append(b)
            
    def forward(self, x, training=True):
        self.activations = [x.reshape(-1, 1)]
        self.z_values = []
        self.dropout_masks = []
        
        current = self.activations[0]
        for i in range(len(self.layer_sizes) - 1):
            z = np.dot(self.weights[i], current) + self.biases[i]
            self.z_values.append(z)
            current = sigmoid(z)
            
            # Apply dropout (except output layer)
            if training and i < len(self.layer_sizes) - 2:
                mask = (np.random.rand(*current.shape) > self.dropout_rate) / (1 - self.dropout_rate)
                current = current * mask
                self.dropout_masks.append(mask)
            
            self.activations.append(current)
            
        return current
    
    def backward(self, y):
        y = y.reshape(-1, 1)
        delta = self.activations[-1] - y
        
        self.dW = []
        self.db = []
        
        mask_idx = 0
        for i in range(len(self.layer_sizes) - 2, -1, -1):
            dW = np.dot(delta, self.activations[i].T)
            db = delta
            
            self.dW.insert(0, dW)
            self.db.insert(0, db)
            
            if i > 0:
                delta = np.dot(self.weights[i].T, delta) * sigmoid_derivative(self.z_values[i-1])
                if mask_idx < len(self.dropout_masks):
                    delta = delta * self.dropout_masks[-(mask_idx + 1)]
                    mask_idx += 1
                
    def update_weights(self):
        for i in range(len(self.layer_sizes) - 1):
            self.weights[i] -= self.lr * self.dW[i]
            self.biases[i] -= self.lr * self.db[i]
            
    def train(self, X, y, epochs=1000):
        for epoch in range(epochs):
            for xi, yi in zip(X, y):
                self.forward(xi, training=True)
                self.backward(yi)
                self.update_weights()
                
    def predict(self, x):
        return self.forward(x, training=False)[0, 0]

## 6. Early Stopping

In [ ]:
def train_with_early_stopping(model, X_train, y_train, X_val, y_val, 
                               max_epochs=5000, patience=50):
    """
    Train with early stopping
    
    Parameters:
    - patience: จำนวน epochs ที่รอก่อนหยุดถ้า validation loss ไม่ลดลง
    """
    best_val_loss = float('inf')
    best_weights = None
    counter = 0
    
    train_losses = []
    val_losses = []
    
    for epoch in range(max_epochs):
        # Train one epoch
        for xi, yi in zip(X_train, y_train):
            model.forward(xi)
            model.backward(yi)
            model.update_weights()
            
        # Compute losses
        train_loss = model.compute_loss(X_train, y_train)
        val_loss = model.compute_loss(X_val, y_val)
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        
        # Check for improvement
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = [w.copy() for w in model.weights]
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break
                
    # Restore best weights
    if best_weights:
        model.weights = best_weights
        
    return train_losses, val_losses

# Add compute_loss method
def compute_loss(self, X, y):
    total = 0
    for xi, yi in zip(X, y):
        pred = self.predict(xi)
        total += (yi - pred)**2
    return total / len(X)

MLP_Dropout.compute_loss = compute_loss

## 7. แบบฝึกหัดการคำนวณ

### แบบฝึกหัดที่ 1: คำนวณ L2 Penalty

In [ ]:
# ให้ weights = [0.5, -0.3, 0.8, 0.2] และ λ = 0.01
# จงคำนวณ L2 penalty

weights = np.array([0.5, -0.3, 0.8, 0.2])
l2_lambda = 0.01

l2_penalty = l2_lambda * np.sum(weights**2)
print(f"Weights: {weights}")
print(f"L2 Penalty (λ=0.01): {l2_penalty:.6f}")

### แบบฝึกหัดที่ 2: คำนวณ L1 Penalty

In [ ]:
# ให้ weights เดียวกัน คำนวณ L1 penalty

l1_lambda = 0.01
l1_penalty = l1_lambda * np.sum(np.abs(weights))
print(f"Weights: {weights}")
print(f"L1 Penalty (λ=0.01): {l1_penalty:.6f}")

### แบบฝึกหัดที่ 3: Dropout Mask

In [ ]:
# ให้ activations = [0.8, 0.3, 0.6, 0.9] และ dropout_rate = 0.5
# จงสร้าง dropout mask และปรับค่า activations

np.random.seed(42)
activations = np.array([[0.8], [0.3], [0.6], [0.9]])
dropout_rate = 0.5

# Create mask
mask = (np.random.rand(*activations.shape) > dropout_rate) / (1 - dropout_rate)
dropped_activations = activations * mask

print(f"Original activations:\n{activations.flatten()}")
print(f"\nDropout mask:\n{mask.flatten()}")
print(f"\nDropped activations:\n{dropped_activations.flatten()}")

### แบบฝึกหัดที่ 4: เปรียบเทียบ Regularization

In [ ]:
# ทดลองฝึกโมเดล 3 แบบและเปรียบเทียบ

# 1. No regularization
mlp1 = MLP_Regularized([2, 32, 1], l1_lambda=0, l2_lambda=0)
mlp1.train(X_train, y_train, epochs=1000)

# 2. L2 regularization
mlp2 = MLP_Regularized([2, 32, 1], l1_lambda=0, l2_lambda=0.01)
mlp2.train(X_train, y_train, epochs=1000)

# 3. L1 regularization
mlp3 = MLP_Regularized([2, 32, 1], l1_lambda=0.01, l2_lambda=0)
mlp3.train(X_train, y_train, epochs=1000)

print("=== Training Loss ===")
print(f"No Reg: {mlp1.compute_loss(X_train, y_train):.4f}")
print(f"L2: {mlp2.compute_loss(X_train, y_train):.4f}")
print(f"L1: {mlp3.compute_loss(X_train, y_train):.4f}")

print("\n=== Test Loss ===")
print(f"No Reg: {mlp1.compute_loss(X_test, y_test):.4f}")
print(f"L2: {mlp2.compute_loss(X_test, y_test):.4f}")
print(f"L1: {mlp3.compute_loss(X_test, y_test):.4f}")

## บทสรุป

Notebook นี้ครอบคลุม:
1. **L1 Regularization**: ทำให้ weights บางตัวเป็น 0 (sparse)
2. **L2 Regularization**: ลดขนาด weights ทุกตัว
3. **Dropout**: สุ่มปิด neurons ระหว่าง training
4. **Early Stopping**: หยุดฝึกเมื่อ validation loss ไม่ลดลง